# Can bashlex parse what agents write?

`mcpath` will expose a virtual shell to agents. Before building the interpreter, we need a **parser**: something that turns a command string into a tree (an *AST*) that our executor can walk.

[bashlex](https://github.com/idank/bashlex) is a ready-made bash parser in Python (it powers explainshell.com). This notebook checks:

1. **Coverage**: how many realistic agent commands does it parse?
2. **Shape**: what does its tree look like, and is it usable for an executor?
3. **Details**: does it keep the information we need (e.g. was a word quoted)?

> Note: bashlex only *parses*. Expansion (`$VAR`, globs, `$(...)`) and execution are our job either way.

In [1]:
import bashlex
from IPython.display import HTML, display

## 1. A corpus of realistic agent commands

Grouped by the shell feature each one exercises. These are the kinds of commands models send to a "run shell command" tool.

In [2]:
COMMANDS = {
    "simple": [
        "ls -la src/",
        "cat README.md",
        "head -n 50 src/main.py",
        "wc -l src/*.py",
    ],
    "quoting": [
        """grep -rn "TODO" --include='*.py' .""",
        """grep -E 'def (foo|bar)\\(' src/app.py""",
        """echo "it's a \\"quoted\\" string" """,
        """find . -name "*.md" -not -path './node_modules/*'""",
    ],
    "pipes & lists": [
        "grep -rn TODO . | wc -l",
        "cat log.txt | sort | uniq -c | sort -rn | head -10",
        "mkdir -p build && cd build && ls",
        "test -f config.toml || echo 'missing config'",
        "cd src; ls; cd ..",
    ],
    "redirections": [
        "ls > files.txt",
        "echo done >> log.txt",
        "grep foo missing.txt 2>&1",
        "grep foo missing.txt 2>/dev/null",
        "wc -l < input.txt",
        "cmd > out.txt 2>&1",
    ],
    "heredoc": [
        "cat > notes.md <<'EOF'\n# Notes\nsome $text here\nEOF\n",
        "cat <<EOF\nhello $USER\nEOF\n",
    ],
    "substitution & vars": [
        'echo "files: $(ls | wc -l)"',
        'NAME=world; echo "hello $NAME"',
        "echo ${HOME}/x",
        "echo `pwd`",
        "echo $((1 + 2))",
    ],
    "control flow": [
        'for f in src/*.py; do echo "$f: $(wc -l < $f)"; done',
        "if [ -d src ]; then ls src; else echo no src; fi",
        'while read line; do echo "> $line"; done < list.txt',
        'case "$x" in *.py) echo py;; *) echo other;; esac',
        "f() { echo hi; }; f",
    ],
    "agent favourites": [
        "find . -name '*.py' -exec grep -l 'import os' {} \\;",
        "find . -type f -name '*.py' | xargs grep -n 'TODO'",
        "ls src/{main,utils}.py",
        "diff <(sort a.txt) <(sort b.txt)",
        "(cd sub && ls)",
        "[[ -f a.txt && -n $x ]] && echo yes",
    ],
}

## 2. Parse everything

`try_parse` returns either the tree or the error. The table is colour-coded: green parsed, red failed.

In [3]:
def try_parse(cmd):
    try:
        return bashlex.parse(cmd), None
    except Exception as e:  # ParsingError, NotImplementedError, ...
        return None, f"{type(e).__name__}: {e}"


results = [
    (group, cmd, *try_parse(cmd))
    for group, cmds in COMMANDS.items()
    for cmd in cmds
]


def show_results(results):
    rows = []
    for group, cmd, trees, err in results:
        ok = err is None
        color = "rgba(46,160,67,.18)" if ok else "rgba(248,81,73,.22)"
        shown = cmd.replace("<", "&lt;").replace("\n", "<b>⏎</b>")
        rows.append(
            f"<tr style='background:{color}'><td>{group}</td>"
            f"<td><code>{shown}</code></td>"
            f"<td>{'✅' if ok else '❌'}</td><td>{err or ''}</td></tr>"
        )
    passed = sum(r[3] is None for r in results)
    display(HTML(
        f"<h3>{passed}/{len(results)} parsed</h3>"
        "<table><tr><th>group</th><th>command</th><th></th><th>error</th></tr>"
        + "".join(rows) + "</table>"
    ))


show_results(results)

group,command,,error
simple,ls -la src/,✅,
simple,cat README.md,✅,
simple,head -n 50 src/main.py,✅,
simple,wc -l src/*.py,✅,
quoting,"grep -rn ""TODO"" --include='*.py' .",✅,
quoting,grep -E 'def (foo|bar)\(' src/app.py,✅,
quoting,"echo ""it's a \""quoted\"" string""",✅,
quoting,"find . -name ""*.md"" -not -path './node_modules/*'",✅,
pipes & lists,grep -rn TODO . | wc -l,✅,
pipes & lists,cat log.txt | sort | uniq -c | sort -rn | head -10,✅,


## 3. What does a tree look like?

Take one realistic command with a pipe, a command substitution and a redirect. `dump()` prints the AST.

Read it top-down:
- **PipelineNode**: two commands joined by `|`
- **CommandNode**: one command; its parts are **WordNode**s (arguments) and **RedirectNode**s
- A word like `$(cat x)` contains a nested **CommandsubstitutionNode**, a full command tree inside a word

Every node has `pos=(start, end)`: its location in the original string. That turns out to be important (section 5).

In [4]:
cmd = 'grep -n "$(cat x)" *.py 2>&1 | wc -l'
print(bashlex.parse(cmd)[0].dump())

PipelineNode(pos=(0, 36), parts=[
  CommandNode(pos=(0, 28), parts=[
    WordNode(pos=(0, 4), word='grep'),
    WordNode(pos=(5, 7), word='-n'),
    WordNode(pos=(8, 18), word='$(cat x)', parts=[
      CommandsubstitutionNode(command=
        CommandNode(pos=(11, 16), parts=[
          WordNode(pos=(11, 14), word='cat'),
          WordNode(pos=(15, 16), word='x'),
        ]), pos=(9, 17)),
    ]),
    WordNode(pos=(19, 23), word='*.py'),
    RedirectNode(input=2, output=1, pos=(24, 28), type='>&'),
  ]),
  PipeNode(pipe='|', pos=(29, 30)),
  CommandNode(pos=(31, 36), parts=[
    WordNode(pos=(31, 33), word='wc'),
    WordNode(pos=(34, 36), word='-l'),
  ]),
])


### Walking the tree

An executor is a function that looks at a node's `kind` and does the right thing. bashlex provides a `nodevisitor` base class, but a tiny recursive function shows the idea more clearly: this is the skeleton our executor will grow from.

In [5]:
def walk(node, depth=0):
    pad = "  " * depth
    label = node.kind
    if node.kind == "word":
        label += f"  {node.word!r}"
    elif node.kind == "redirect":
        target = getattr(node.output, "word", node.output)  # file name, or fd number for 2>&1
        label += f"  {node.type!r} -> {target!r}"
    elif node.kind in ("pipe", "operator"):
        label += f"  {getattr(node, 'pipe', None) or node.op!r}"
    print(pad + label)
    for child in getattr(node, "parts", []) or getattr(node, "list", []):
        walk(child, depth + 1)
    if node.kind == "commandsubstitution":
        walk(node.command, depth + 1)


for tree in bashlex.parse('mkdir -p out && grep -rn TODO . | sort > out/todos.txt'):
    walk(tree)

list
  command
    word  'mkdir'
    word  '-p'
    word  'out'
  operator  '&&'
  pipeline
    command
      word  'grep'
      word  '-rn'
      word  'TODO'
      word  '.'
    pipe  '|'
    command
      word  'sort'
      redirect  '>' -> 'out/todos.txt'


## 4. Looking at the failures

Four commands failed. Are they dealbreakers?

### 4a. Quoted heredoc delimiters: a real bug

Agents write files with `cat > file <<'EOF' ... EOF` **constantly**. Quoting the delimiter (`'EOF'`) means "don't expand `$vars` inside". bashlex fails on it, because it searches for a line containing the quotes literally (`'EOF'`) instead of `EOF`. Let's confirm it's only the quoted form:

In [6]:
variants = {
    "unquoted  <<EOF":    "cat <<EOF\nhi\nEOF\n",
    "single    <<'EOF'":  "cat <<'EOF'\nhi\nEOF\n",
    'double    <<"EOF"':  'cat <<"EOF"\nhi\nEOF\n',
    "tab-strip <<-EOF":   "cat <<-EOF\n\thi\n\tEOF\n",
}
for name, c in variants.items():
    _, err = try_parse(c)
    print(f"{name:22} {'ok' if err is None else 'FAIL  ' + err}")

unquoted  <<EOF        ok
single    <<'EOF'      FAIL  ParsingError: here-document at line 0 delimited by end-of-file (wanted "'EOF'") (position 19)
double    <<"EOF"      FAIL  ParsingError: here-document at line 0 delimited by end-of-file (wanted '"EOF"') (position 19)
tab-strip <<-EOF       ok


### 4b. The other three

| Failure | How often do agents use it? | Impact |
|---|---|---|
| `$((1 + 2))` arithmetic | Rare | Low; clear error message |
| `case ... esac` | Rare | Low |
| `[[ ... ]]` test | Occasional | Medium; `[ ... ]` works, so we can tell the model to use it |

The heredoc bug is the one that matters: it's the #1 way agents write files through a shell.

## 5. Is quoting information preserved?

In bash, `*.md` expands to matching files but `"*.py"` stays literal (so `find -name "*.py"` receives the pattern, not a file list). The executor must know which words were quoted.

bashlex's `.word` has the quotes **removed**, so it looks the same either way. But `pos` points back into the original string, so we can recover it:

In [7]:
s = """find . -name "*.py" *.md 'a b'"""
print(f"{'word (bashlex)':16} {'source slice':16} quoted?")
for w in bashlex.parse(s)[0].parts:
    src = s[w.pos[0]:w.pos[1]]
    print(f"{w.word!r:16} {src!r:16} {src != w.word}")

word (bashlex)   source slice     quoted?
'find'           'find'           False
'.'              '.'              False
'-name'          '-name'          False
'*.py'           '"*.py"'         True
'*.md'           '*.md'           False
'a b'            "'a b'"          True


So quoting is recoverable, but only through this `pos` trick. We'd have to re-scan the source slice ourselves to know *which parts* of a word were quoted (e.g. `"$dir"/*.py` is half quoted). That's a sign that expansion logic will fight bashlex a bit.

## 6. Verdict

**What works:** 33/37. Pipes, lists, redirections, `$(...)`, variables, `for`/`if`/`while`, functions, subshells, process substitution. The AST is clean and easy to walk.

**What doesn't:**
- ❌ quoted heredocs (`<<'EOF'`): a common agent pattern, a real problem
- ❌ `[[ ]]`, `case`, `$(( ))`: rare, acceptable with clear errors
- ⚠️ quote info is only recoverable through `pos` + re-scanning

**Options:**
1. **Use bashlex behind a `Parser` interface** and work around the heredoc bug (e.g. rewrite `<<'EOF'` → `<<EOF` before parsing and remember it was quoted). Fast start.
2. **Write our own parser.** More work; full control over quoting and heredocs, and a great way to learn how a shell works.

Either way, keep the rest of the code depending on **our own AST types**, not bashlex's, so switching later is cheap.